In [5]:
import pandas as pd
import sqlite3

df_ev = pd.read_csv('../data/processed/ev_catalog_clean.csv')
conn = sqlite3.connect('../data/processed/ev_catalog.db')
df_ev.to_sql('ev_models', conn, if_exists='replace', index=False)

#Evolución del mercado EV/año
q_evolucion = """
SELECT
    model_year,
    COUNT(*) AS total_models,
    ROUND(AVG(range_km), 1) AS avg_range_km,
    ROUND(AVG(motor_power_kw), 1) AS avg_power_kw,
    ROUND(AVG(comb_kwh_100km), 2) AS avg_consumption_kwh
FROM ev_models
GROUP BY model_year
ORDER BY model_year ASC;
"""
#Marcas con mejor autonomía promedio
q_marcas = """
SELECT
    brand,
    COUNT(*) AS models_count,
    MAX(range_km) AS max_range_km,
    ROUND(AVG(range_km), 1) AS avg_range_km,
    ROUND(AVG(km_charged_per_hour), 1) AS avg_charging_speed_kmh
FROM ev_models
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY avg_range_km DESC;
"""

#Ranking de los modelos con consumo más eficiente
q_eficiencia = """
WITH RankedModels AS (
    SELECT
        vehicle_size,
        brand,
        model,
        model_year,
        comb_kwh_100km,
        range_km,
        RANK() OVER (PARTITION BY vehicle_size ORDER BY comb_kwh_100km ASC, range_km DESC) AS efficiency_rank
    FROM ev_models
)
SELECT *
FROM RankedModels
WHERE efficiency_rank <= 2
ORDER BY vehicle_size, efficiency_rank;
"""

#Comparativa enfocada en Ford contra el promedio del mercado
q_ford = """
SELECT
    model_year,
    model,
    vehicle_size,
    motor_power_kw,
    range_km,
    comb_kwh_100km,
    charging_time_h
FROM ev_models
WHERE brand = 'Ford'
ORDER BY model_year DESC, range_km DESC;
"""

print("--- 1. Evolución anual ---")
display(pd.read_sql_query(q_evolucion, conn))

print("--- 2. Marcas con mejor autonomía ---")
display(pd.read_sql_query(q_marcas, conn))

print("--- 3. Ranking de eficiencia ---")
display(pd.read_sql_query(q_eficiencia, conn))

print("--- 4. Catálogo de Ford ---")
display(pd.read_sql_query(q_ford, conn))

--- 1. Evolución anual ---


,model_year,total_models,avg_range_km,avg_power_kw,avg_consumption_kwh
0,2012,2,108.5,64.5,19.90
1,2013,9,218.7,158.4,21.14
2,2014,9,210.3,135.0,20.36
3,2015,14,239.4,181.4,20.29
4,2016,19,276.2,252.3,20.59


--- 2. Marcas con mejor autonomía ---


,brand,models_count,max_range_km,avg_range_km,avg_charging_speed_kmh
0,Tesla,23,473,393.7,34.6
1,Nissan,6,172,135.2,23.8
2,Chevrolet,3,131,131.0,18.7
3,Ford,4,122,122.0,30.5
4,Smart,8,109,109.0,13.6
5,Mitsubishi,5,100,100.0,14.3


--- 3. Ranking de eficiencia ---


,vehicle_size,brand,model,model_year,comb_kwh_100km,range_km,efficiency_rank
0,COMPACT,Ford,FOCUS ELECTRIC,2013,20.0,122,1
1,COMPACT,Ford,FOCUS ELECTRIC,2014,20.0,122,1
2,COMPACT,Ford,FOCUS ELECTRIC,2015,20.0,122,1
3,COMPACT,Ford,FOCUS ELECTRIC,2016,20.0,122,1
4,FULL-SIZE,Tesla,MODEL S 90D (Refresh),2016,20.3,473,1
5,FULL-SIZE,Tesla,MODEL S 70D,2015,20.7,386,2
6,FULL-SIZE,Tesla,MODEL S 70D,2016,20.7,386,2
7,MID-SIZE,Nissan,LEAF,2014,18.4,135,1
8,MID-SIZE,Nissan,LEAF,2015,18.4,135,1
9,MID-SIZE,Nissan,LEAF (24 kWh battery),2016,18.4,135,1


--- 4. Catálogo de Ford ---


,model_year,model,vehicle_size,motor_power_kw,range_km,comb_kwh_100km,charging_time_h
0,2016,FOCUS ELECTRIC,COMPACT,107,122,20.0,4
1,2015,FOCUS ELECTRIC,COMPACT,107,122,20.0,4
2,2014,FOCUS ELECTRIC,COMPACT,107,122,20.0,4
3,2013,FOCUS ELECTRIC,COMPACT,107,122,20.0,4
